# Stage 1 Training + Diagnostics Notebook

This notebook keeps a **single Stage-1 entry point**:

```bash
python scripts/train_stage1.py
```

Training, smoke tests, and diagnostics all call the same script. Diagnostics save PNG figures, CSV/JSON statistics, and a zip archive under the run folder. Edit `/path/to/PartImageNet` and checkpoint paths before running.

## 1. Check PartImageNet layout

This verifies that the annotation JSONs and image folders resolve. It does not train.

In [ ]:
!python scripts/train_stage1.py \
  --config configs/stage1_quality_upgrade.yaml \
  --partimagenet-root /path/to/PartImageNet \
  --print-data-layout

## 2. Smoke-test the exact Stage-1 setup

This runs one forward/audit pass with the high-resolution refinement and quality-loss configuration. Use `--allow-partial-load` when warm-starting from an older Stage-1 checkpoint that does not contain the high-resolution refinement layers.

In [ ]:
!python scripts/train_stage1.py \
  --config configs/stage1_quality_upgrade.yaml \
  --device auto \
  --partimagenet-root /path/to/PartImageNet \
  --save-dir runs/stage1_quality_upgrade \
  --warm-start runs/stage1/checkpoints/stage1_best.pt \
  --allow-partial-load \
  --quality-loss \
  --use-highres-refine \
  --presence-threshold 0.25 \
  --topk-presence-k 128 \
  --small-part-area-tau 0.015 \
  --small-part-weight-max 6.0 \
  --small-part-weight-power 0.5 \
  --smoke-only

## 3. Full Stage-1 training

This is the main training command. It saves:

```text
runs/stage1_quality_upgrade/checkpoints/stage1_best.pt
runs/stage1_quality_upgrade/checkpoints/stage1_last.pt
runs/stage1_quality_upgrade/stage1_history.csv
runs/stage1_quality_upgrade/stage1_history.json
```

The final flag `--diagnostics-after-training` automatically reloads the best checkpoint and saves diagnostic figures/statistics after training.

In [ ]:
!python scripts/train_stage1.py \
  --config configs/stage1_quality_upgrade.yaml \
  --device auto \
  --partimagenet-root /path/to/PartImageNet \
  --save-dir runs/stage1_quality_upgrade \
  --warm-start runs/stage1/checkpoints/stage1_best.pt \
  --allow-partial-load \
  --quality-loss \
  --use-highres-refine \
  --epochs 24 \
  --batch-size 8 \
  --lr 5e-5 \
  --presence-threshold 0.25 \
  --topk-presence-k 128 \
  --small-part-area-tau 0.015 \
  --small-part-weight-max 6.0 \
  --small-part-weight-power 0.5 \
  --quality-presence-bce 0.40 \
  --valid-absent-topmean-fp 0.08 \
  --valid-absent-mean-fp 0.02 \
  --invalid-part-topmean 0.35 \
  --invalid-part-mean 0.08 \
  --gt-support-leak 0.35 \
  --pred-support-containment 0.25 \
  --boundary-loss 0.08 \
  --focal-functional 0.12 \
  --tversky-functional 0.12 \
  --quality-topq 0.02 \
  --diagnostics-after-training \
  --diag-save-dir runs/stage1_quality_upgrade/diagnostics_stage1 \
  --diag-balanced-samples-per-class 8 \
  --diag-max-batches 50 \
  --diag-num-samples 12 \
  --diag-max-parts-per-sample 8 \
  --diag-mask-threshold 0.40

## 4. Diagnostics-only run for an existing checkpoint

Run this after training, or whenever you want to inspect a checkpoint without training. It saves local figures and tables to:

```text
runs/stage1_quality_upgrade/diagnostics_stage1/
runs/stage1_quality_upgrade/diagnostics_stage1.zip
```

Key outputs include:

```text
figures/stage1_training_curves.png
figures/stage1_per_part_quality.png
figures/stage1_per_class_part_iou_heatmap.png
figures/stage1_per_class_part_presence_f1_heatmap.png
figures/stage1_presence_threshold_sweep.png
figures/samples/stage1_sample_*.png
tables/stage1_per_part.csv
tables/stage1_per_class_part.csv
tables/presence_threshold_sweep.csv
hkg_part_quality_hint.json
```

In [ ]:
!python scripts/train_stage1.py \
  --config configs/stage1_quality_upgrade.yaml \
  --device auto \
  --partimagenet-root /path/to/PartImageNet \
  --save-dir runs/stage1_quality_upgrade \
  --diag-checkpoint runs/stage1_quality_upgrade/checkpoints/stage1_best.pt \
  --allow-partial-load \
  --quality-loss \
  --use-highres-refine \
  --presence-threshold 0.25 \
  --topk-presence-k 128 \
  --diagnostics-only \
  --diag-save-dir runs/stage1_quality_upgrade/diagnostics_stage1 \
  --diag-balanced-samples-per-class 8 \
  --diag-max-batches 50 \
  --diag-num-samples 12 \
  --diag-max-parts-per-sample 8 \
  --diag-mask-threshold 0.40 \
  --diag-presence-thresholds 0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50

## 5. Fast diagnostics on a small subset

Use this when iterating on Stage-1 settings. It runs quickly and still saves figures locally.

In [ ]:
!python scripts/train_stage1.py \
  --config configs/stage1_quality_upgrade.yaml \
  --device auto \
  --partimagenet-root /path/to/PartImageNet \
  --save-dir runs/stage1_quality_upgrade \
  --diag-checkpoint runs/stage1_quality_upgrade/checkpoints/stage1_best.pt \
  --allow-partial-load \
  --quality-loss \
  --use-highres-refine \
  --presence-threshold 0.25 \
  --topk-presence-k 128 \
  --diagnostics-only \
  --diag-save-dir runs/stage1_quality_upgrade/diagnostics_stage1_fast \
  --diag-balanced-samples-per-class 2 \
  --diag-max-batches 8 \
  --diag-num-samples 6 \
  --diag-max-parts-per-sample 8 \
  --diag-mask-threshold 0.40

## 6. Optional: train from scratch

Use this only if no warm-start checkpoint exists. It still uses the same Stage-1 entry point.

In [ ]:
!python scripts/train_stage1.py \
  --config configs/stage1_quality_upgrade.yaml \
  --device auto \
  --partimagenet-root /path/to/PartImageNet \
  --save-dir runs/stage1_quality_upgrade_from_scratch \
  --quality-loss \
  --use-highres-refine \
  --epochs 24 \
  --batch-size 8 \
  --lr 5e-5 \
  --diagnostics-after-training \
  --diag-save-dir runs/stage1_quality_upgrade_from_scratch/diagnostics_stage1 \
  --diag-balanced-samples-per-class 8 \
  --diag-max-batches 50 \
  --diag-num-samples 12